In [ ]:
'''
This is an optimization program that takes an excel sheet of titles from GOBI and creates a list of suggested titles to purchase 
based on our input criteria. It uses the PuLP library for optimization and Pandas for data handling. It was written with the help of tutorials from the following sources:
https://github.com/coin-or/pulp/blob/master/doc/pulp.pdf
https://pyomo-simplemodel.readthedocs.io/en/latest/knapsack.html
https://pypi.org/project/PuLP/

last edited: 9/18/2026

'''

The first step is to load the data as a Pandas DataFrame and add the scores for year and press type. This can be tricky because the columns do not always have the same name.
Note that the index column is the EBSCO Product ID. It is important to establish this index column. It will be used to create a binary variable to help the solver choose between titles. I also establish a budget variable here, and note that it is in a list. You may add several budgets and run this as a function. 

In [1]:
import pandas as pd
from pulp import *

#create df from input spreadsheet
data_in = "BulkEbookPurchase - Copy (2).xlsx"
df = pd.read_excel(data_in , sheet_name = "DataIn", index_col = 'EBSCO Product ID')

#set budget vars, you may create a list of budgets here and introduce an iterative process that uses a Python function
budget = [10000]

#the press scores dictionary
presses = {"US University Press": 4.0, 
           "Other Univ or Soc Press" : 2.0,
           "Other Press" : 0.1}
#df.head()

df["Press_Priority"] = df["Press Type"].map(presses)

df["EBSCO 1U List Price"] = pd.to_numeric(df["EBSCO 1U List Price"], errors="coerce")
df.dropna(subset=["EBSCO 1U List Price"], inplace=True)

year_score_dict = {2027: 3.2, 2026: 1.6, 2025: 0.8, 2024: 0.4, 2023: 0.2, 2022: 0.1}

df ["Year_Score"]  = df["PUBLICATION YEAR"].map(year_score_dict)

#add priority score and year score to create an overall scores for each title
df["Score"] = df["Press_Priority"] + df["Year_Score"]

#this is for another way you might use the program with call numbers
# df['LC_Class_FirstLetter'] = df['LC Call Number'].str[0]
# p_rows = df.index[df['LC_Class_FirstLetter'] == "P"]
# l_rows = df.index[df['LC_Class_FirstLetter']== "L"]

#I check for nulls in the Score column here to make sure the mapping worked
print(df[df["Score"].isna()])


Empty DataFrame
Columns: [Customer owns print, In customer's DDA pool, EBSCO eISBN, EBSCO 1U List Price, GOBI Sales (25 or more copies sold), GOBI Sales (50 or more copies sold), GOBI Sales (100 or more copies sold), GOBI Titlelink Number, CUSTOMER Acct, Date of approval slip, Customer fund code, TITLE, Author, PUBLICATION YEAR, GOBI Imprint ID, Imprint_Name, Publisher Name, Subject, Bib Notes, Pagination, LC Class, LC Class Number, Full_Lc_Call_Number, Lc_Class_Sort, GOBI Content Level, GOBI Select Level, Aspect, Interdisciplinary, Alt_Edition, Country of Publication, Co_Publisher, Distributed, Edition statement, Edition_Type, Faculty, Format, Geo Country, Geo Region, Literature_Type, Language, Related to, Reprint, Reference, Series_Id, Series_Title, Series Type, First Appearing Series Volume, Translation, US University Press, Other University or Society Press, Other Press, Press Type, Press_Priority, Year_Score, Score]
Index: []

[0 rows x 55 columns]


Next, you need to set up the problem. PuLP has extensive documentation about how to do this, so I will not cover every detail. However, note that the problem variable creates a class that will maximize the value. I have established the binary decision variable in the variable title_vars, and created an upper limit for budget overspend at $2. I've commented out constraint two, which shows a way you might use this program to control the mixture of LC class numbers. We do not have time to go over that use case in this presentation.

In [2]:
#Create the model/problem instance. "TitleList" is the name here, but it can be named anything with no spaces.  
#You are essentially creating a problem class that takes the objective function, constraints and variables. source: (https://pyomo-simplemodel.readthedocs.io/en/latest/knapsack.html) 
problem = LpProblem("TitleList", LpMaximize)

# Create decision variables from your index column, since this is a knapsack problem, the category is binary source: (https://www.coin-or.org/PuLP/pulp.html)
title_vars = LpVariable.dicts("x", df.index, lowBound=0, cat=LpBinary)
#the upBound controls the amount the program will go outside of the budget (boundary $2)
overspend = LpVariable("overspend", lowBound=0, upBound=2)
underspend = LpVariable("underspend", lowBound=0)
total_score = lpSum(df.loc[i, "Score"] * title_vars[i] for i in df.index) 
total_cost = lpSum(df.loc[i, "EBSCO 1U List Price"] * title_vars[i] for i in df.index)

# Objective: maximize total score (press priority + year score) and subtract for overspend or underspend the boundary is at $2
problem += ( total_score -(0.01 * overspend) - (0.01 * underspend))

#example of objective function to only maximize titles
#problem += lpSum(title_vars[i] for i in df.index)

#Constraint 1: Total price should not exceed the budget
problem += total_cost <= (budget + overspend), "Soft Budget Over"
problem += total_cost <= (budget + underspend), "Soft Budget Under"

#Constraint 2: 20% titles L 15% P
# problem += lpSum(title_vars[i] for i in l_rows) >= .20 * lpSum(title_vars[i] for i in df.index)
# problem += lpSum(title_vars[i] for i in p_rows) >= .15 * lpSum(title_vars[i] for i in df.index)
  
try:
    problem.solve()
except PulpSolverError:
    print("Solver failed.")

selected_indices = [i for i in df.index if title_vars[i].varValue == 1]
selected_df = df.loc[selected_indices].copy()
total_cost = selected_df["EBSCO 1U List Price"].sum()
avg_price = selected_df["EBSCO 1U List Price"].mean()
print(f"Selected {len(selected_df)} titles for budget ${budget}")
print(f"Total cost: ${total_cost:.2f} | Average price: ${avg_price:.2f}")

print("Done")
selected_df.head()

selected_df.to_excel("PulpOutputCharleston10K.xlsx")

Selected 287 titles for budget $[10000]
Total cost: $10001.14 | Average price: $34.85
Done


Next, I created a pivot table so we can look at the output more closely. You may also add a pivot table to your Excel sheet if you would like to make the program fancier.

In [3]:
pivot = pd.pivot_table(selected_df, index="PUBLICATION YEAR", columns="Press Type", values="TITLE", aggfunc="count", fill_value = 0)

print(pivot)

Press Type        Other Press  Other Univ or Soc Press  US University Press
PUBLICATION YEAR                                                           
2022                        0                        7                   53
2023                        1                        6                   14
2024                        1                        8                   34
2025                        9                        9                   61
2026                       19                        8                   57
